## ToyAIKit Demo

This python library helps automates some of the manual processes I wrote in teh previous agentic loop notebook

In [21]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import os
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [5]:
# Create search function
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [27]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
# Define Search Tool manually
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [7]:
# Registering the search function tool
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [8]:
# Creating a function that will write our search tool schema automatically
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Note: the type hints and dict help set the metadata that will be repurposed to output the tool json schema needed for the LLM. The toyaikit utilizes the python inspect library (standard in frameworks) to extract and define the parameters needed to create the json schema for the search tool or any tool.

In [9]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [10]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [22]:
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

model = "qwen/qwen3.5-9b"

In [23]:
# Developer Prompt - we define how we want the LLM model to behave
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):

1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.
   This will give you general information about the topic.

2. ANALYZE: Carefully read all results from the first search.
   
3. SECOND SEARCH: Based on what you found, perform a SECOND search with:
   - New keywords derived from the first results
   - More specific questions or follow-up queries
   - OR ask yourself: "What else should I search for to give a complete answer?"

4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.

5. ASK: End by asking if there are other areas the user wants to explore.

Make sure you perform at least 2 separate searches before answering.

Make sure you perform no more than 3 separate searches before answering.
""".strip()

In [25]:
# Creating the chat interface and a callback, then build the runner
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)


runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=model, client=openai_client)
)

In [28]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/01_module_agentic_rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [30]:
result.all_messages

[EasyInputMessage(content='You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.', role='developer', pha

In [33]:
# Now we will take the messages from the previous result and pass them as previous_messages on the next loop call:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/01_module_agentic_rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [34]:
# Interactive chat
runner.run()

-> Response received


-> Response received


-> Response received


/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/01_module_agentic_rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


-> Response received


...

-> Response received


/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/01_module_agentic_rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


Chat ended.


/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/01_module_agentic_rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:184: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(


LoopResult(new_messages=[EasyInputMessage(content='You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.